In [1]:
# DVF1-1 — Téléchargement des données Île-de-France
import urllib.request
import os

# Dossier où on va stocker les données
os.makedirs("data", exist_ok=True)

url = "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/75.csv.gz"
fichiers = {
    "75_paris.csv.gz":    "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/75.csv.gz",
    "92_hauts_de_seine.csv.gz": "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/92.csv.gz",
    "93_seine_saint_denis.csv.gz": "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/93.csv.gz",
    "94_val_de_marne.csv.gz": "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/94.csv.gz",
    "77_seine_et_marne.csv.gz": "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/77.csv.gz",
    "78_yvelines.csv.gz":  "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/78.csv.gz",
    "91_essonne.csv.gz":   "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/91.csv.gz",
    "95_val_d_oise.csv.gz":"https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/95.csv.gz",
}

for nom_fichier, lien in fichiers.items():
    chemin = f"data/{nom_fichier}"
    if os.path.exists(chemin):
        print(f"✓ Déjà téléchargé : {nom_fichier}")
    else:
        print(f"⬇ Téléchargement : {nom_fichier}...")
        urllib.request.urlretrieve(lien, chemin)
        print(f"✓ OK")

print("\n✅ Tous les fichiers sont prêts dans le dossier data/")

⬇ Téléchargement : 75_paris.csv.gz...
✓ OK
⬇ Téléchargement : 92_hauts_de_seine.csv.gz...
✓ OK
⬇ Téléchargement : 93_seine_saint_denis.csv.gz...
✓ OK
⬇ Téléchargement : 94_val_de_marne.csv.gz...
✓ OK
⬇ Téléchargement : 77_seine_et_marne.csv.gz...
✓ OK
⬇ Téléchargement : 78_yvelines.csv.gz...
✓ OK
⬇ Téléchargement : 91_essonne.csv.gz...
✓ OK
⬇ Téléchargement : 95_val_d_oise.csv.gz...
✓ OK

✅ Tous les fichiers sont prêts dans le dossier data/


In [2]:
# DVF1-2 — Inspection du schéma
import pandas as pd

df = pd.read_csv("data/75_paris.csv.gz", compression="gzip", low_memory=False)

# --- Taille du fichier ---
print("=== TAILLE ===")
print(f"Nombre de lignes    : {len(df):,}")
print(f"Nombre de colonnes  : {len(df.columns)}")

# --- Noms des colonnes ---
print("\n=== COLONNES ===")
for col in df.columns:
    print(f"  {col}")

# --- Aperçu des 3 premières lignes ---
print("\n=== APERÇU ===")
df.head(3)


=== TAILLE ===
Nombre de lignes    : 74,800
Nombre de colonnes  : 40

=== COLONNES ===
  id_mutation
  date_mutation
  numero_disposition
  nature_mutation
  valeur_fonciere
  adresse_numero
  adresse_suffixe
  adresse_nom_voie
  adresse_code_voie
  code_postal
  code_commune
  nom_commune
  code_departement
  ancien_code_commune
  ancien_nom_commune
  id_parcelle
  ancien_id_parcelle
  numero_volume
  lot1_numero
  lot1_surface_carrez
  lot2_numero
  lot2_surface_carrez
  lot3_numero
  lot3_surface_carrez
  lot4_numero
  lot4_surface_carrez
  lot5_numero
  lot5_surface_carrez
  nombre_lots
  code_type_local
  type_local
  surface_reelle_bati
  nombre_pieces_principales
  code_nature_culture
  nature_culture
  code_nature_culture_speciale
  nature_culture_speciale
  surface_terrain
  longitude
  latitude

=== APERÇU ===


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2024-1193218,2024-01-04,1,Vente,1042000.0,4.0,NaN,VLA PERREUR,7288,75020.0,...,Appartement,86.0,4.0,NaN,NaN,NaN,NaN,NaN,2.405228,48.868216
1,2024-1193218,2024-01-04,1,Vente,1042000.0,4.0,NaN,VLA PERREUR,7288,75020.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.405228,48.868216
2,2024-1193218,2024-01-04,1,Vente,1042000.0,16.0,NaN,RUE DE LA DHUIS,2786,75020.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.405228,48.868216


In [3]:
# DVF1-3 — Nettoyage des données

# 1. Garder uniquement les ventes simples
df = df[df["nature_mutation"] == "Vente"]
print(f"Après filtre ventes : {len(df):,} lignes")

# 2. Garder uniquement appartements et maisons
df = df[df["type_local"].isin(["Appartement", "Maison"])]
print(f"Après filtre type   : {len(df):,} lignes")

# 3. Supprimer les lignes sans prix ou sans surface
df = df.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
print(f"Après suppression NaN : {len(df):,} lignes")

# 4. Supprimer les surfaces aberrantes (moins de 9m² ou plus de 500m²)
df = df[(df["surface_reelle_bati"] >= 9) & (df["surface_reelle_bati"] <= 500)]
print(f"Après filtre surface : {len(df):,} lignes")

# 5. Calculer le prix au m²
df["prix_m2"] = df["valeur_fonciere"] / df["surface_reelle_bati"]

# 6. Supprimer les prix au m² aberrants (moins de 1000€ ou plus de 50 000€)
df = df[(df["prix_m2"] >= 1000) & (df["prix_m2"] <= 50000)]
print(f"Après filtre prix/m² : {len(df):,} lignes")

print(f"\n✅ Données propres : {len(df):,} transactions")
df.head(3)

Après filtre ventes : 73,598 lignes
Après filtre type   : 32,910 lignes
Après suppression NaN : 32,862 lignes
Après filtre surface : 32,549 lignes
Après filtre prix/m² : 27,912 lignes

✅ Données propres : 27,912 transactions


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,prix_m2
0,2024-1193218,2024-01-04,1,Vente,1042000.0,4.0,NaN,VLA PERREUR,7288,75020.0,...,86.0,4.0,NaN,NaN,NaN,NaN,NaN,2.405228,48.868216,12116.279070
5,2024-1193219,2024-01-04,1,Vente,299120.0,17.0,B,RUE DU ROI D ALGER,8314,75018.0,...,36.0,3.0,NaN,NaN,NaN,NaN,NaN,2.347665,48.895338,8308.888889
6,2024-1193220,2024-01-03,1,Vente,426471.0,166.0,NaN,AV PARMENTIER,7067,75010.0,...,42.0,1.0,NaN,NaN,NaN,NaN,NaN,2.370282,48.871532,10154.071429


In [4]:
# DVF1-4 — Chargement et fusion de toute l'Île-de-France

import pandas as pd
import os

# Tous nos fichiers avec leur département
fichiers = {
    "75": "data/75_paris.csv.gz",
    "92": "data/92_hauts_de_seine.csv.gz",
    "93": "data/93_seine_saint_denis.csv.gz",
    "94": "data/94_val_de_marne.csv.gz",
    "77": "data/77_seine_et_marne.csv.gz",
    "78": "data/78_yvelines.csv.gz",
    "91": "data/91_essonne.csv.gz",
    "95": "data/95_val_d_oise.csv.gz",
}

def nettoyer(df, dept):
    """Applique le même nettoyage à n'importe quel département"""
    df = df[df["nature_mutation"] == "Vente"]
    df = df[df["type_local"].isin(["Appartement", "Maison"])]
    df = df.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
    df = df[(df["surface_reelle_bati"] >= 9) & (df["surface_reelle_bati"] <= 500)]
    df["prix_m2"] = df["valeur_fonciere"] / df["surface_reelle_bati"]
    df = df[(df["prix_m2"] >= 1000) & (df["prix_m2"] <= 50000)]
    df["departement"] = dept  # on ajoute le numéro de département
    return df

# On charge et nettoie chaque département, on stocke dans une liste
tous_les_df = []

for dept, chemin in fichiers.items():
    print(f"⏳ Chargement département {dept}...")
    df_dept = pd.read_csv(chemin, compression="gzip", low_memory=False)
    df_dept = nettoyer(df_dept, dept)
    tous_les_df.append(df_dept)
    print(f"✓ {dept} — {len(df_dept):,} transactions propres")

# On fusionne tout en un seul tableau
print("\n🔄 Fusion de tous les départements...")
df_idf = pd.concat(tous_les_df, ignore_index=True)

print(f"\n✅ Île-de-France complète : {len(df_idf):,} transactions")
print(f"   Départements : {df_idf['departement'].unique()}")

⏳ Chargement département 75...
✓ 75 — 27,912 transactions propres
⏳ Chargement département 92...
✓ 92 — 16,274 transactions propres
⏳ Chargement département 93...
✓ 93 — 13,892 transactions propres
⏳ Chargement département 94...
✓ 94 — 13,510 transactions propres
⏳ Chargement département 77...
✓ 77 — 16,509 transactions propres
⏳ Chargement département 78...
✓ 78 — 14,501 transactions propres
⏳ Chargement département 91...
✓ 91 — 13,201 transactions propres
⏳ Chargement département 95...
✓ 95 — 11,667 transactions propres

🔄 Fusion de tous les départements...

✅ Île-de-France complète : 127,466 transactions
   Départements : ['75' '92' '93' '94' '77' '78' '91' '95']


In [5]:
# DVF1-5 — Export du fichier propre

# On garde uniquement les colonnes utiles pour la suite
colonnes_utiles = [
    "id_mutation", "date_mutation", "nature_mutation",
    "valeur_fonciere", "type_local", "surface_reelle_bati",
    "nombre_pieces_principales", "adresse_nom_voie",
    "code_postal", "nom_commune", "code_departement",
    "departement", "longitude", "latitude", "prix_m2"
]

df_final = df_idf[colonnes_utiles].copy()

# Export en CSV
df_final.to_csv("data/idf_propre.csv", index=False)

# Vérification
taille = os.path.getsize("data/idf_propre.csv") / 1024 / 1024
print(f"✅ Fichier sauvegardé : data/idf_propre.csv")
print(f"   Taille : {taille:.1f} Mo")
print(f"   Lignes : {len(df_final):,}")
print(f"   Colonnes : {len(df_final.columns)}")
print(f"\nAperçu :")
df_final.head(3)

✅ Fichier sauvegardé : data/idf_propre.csv
   Taille : 17.2 Mo
   Lignes : 127,466
   Colonnes : 15

Aperçu :


,id_mutation,date_mutation,nature_mutation,valeur_fonciere,type_local,surface_reelle_bati,nombre_pieces_principales,adresse_nom_voie,code_postal,nom_commune,code_departement,departement,longitude,latitude,prix_m2
0,2024-1193218,2024-01-04,Vente,1042000.0,Appartement,86.0,4.0,VLA PERREUR,75020.0,Paris 20e Arrondissement,75,75,2.405228,48.868216,12116.279070
1,2024-1193219,2024-01-04,Vente,299120.0,Appartement,36.0,3.0,RUE DU ROI D ALGER,75018.0,Paris 18e Arrondissement,75,75,2.347665,48.895338,8308.888889
2,2024-1193220,2024-01-03,Vente,426471.0,Appartement,42.0,1.0,AV PARMENTIER,75010.0,Paris 10e Arrondissement,75,75,2.370282,48.871532,10154.071429
